# Surgical Instrument Force & Motion Analysis

Analyzes sequence recordings (`*.igs.mha`, IGSIO/PLUS metafile format) of surgical
training trials, organized **one subfolder per participant**:

```
data/
  P01/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
  P02/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
  ...
```

Each file contains, per frame:

| Field | Meaning |
|---|---|
| `*TipToWorldTransform` | 4×4 pose of each instrument tip (**Bipolar**, **Cavitron**, **Scissors**) |
| `Force` | `fx fy fz tx ty tz` — 3D force + 3D torque from the force sensor |
| `BipolarCollectedPoint0..3` | 4 fiducial points used to register trials into a common frame |
| `Timestamp` | frame time in seconds |

**What this notebook produces**

1. **Point-wise rigid registration** of every trial onto a single common reference
   frame, using the 4 collected fiducials (all trials, all participants share it).
2. **Time normalized to [0, 1]** per trial so recordings of different length align.
3. **Force magnitude** `√(fx²+fy²+fz²)` — one panel per participant.
4. **Velocity, acceleration, jerk** per instrument per trial — one figure per participant.
5. **3D trajectories** as time-colored lines — one figure per participant (trials × instruments).
6. **Summative figures**: per-participant averages and a cross-participant comparison
   of average force / velocity / acceleration / jerk.

> Runs as-is in **Google Colab**. Upload your participant folders (or a zip) when
> prompted, or mount Google Drive and point `DATA_DIR` at the folder that holds them.

## 1 · Setup

In [ ]:
# Colab already ships numpy / scipy / matplotlib; this is a no-op there and a
# convenience when running elsewhere.
import importlib, subprocess, sys
for pkg in ("numpy", "scipy", "matplotlib", "pandas"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, re, glob
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# ---- Plot style ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "legend.fontsize": 9, "legend.frameon": False,
    "font.size": 10,
})

INSTRUMENTS = ["Bipolar", "Cavitron", "Scissors"]
# Okabe-Ito colorblind-safe palette, fixed order per instrument.
INST_COLOR = {"Bipolar": "#0072B2", "Cavitron": "#E69F00", "Scissors": "#009E73"}
# Sequential map used everywhere "value = time".
TIME_CMAP = "viridis"
print("Setup complete.")

## 2 · Locate the data

Set `DATA_DIR` to the folder that holds the **participant subfolders**. Each
subfolder (its name is the participant id) should contain that participant's trial
`.igs.mha` files.

- **Colab, quick upload** — leave `DATA_DIR = "data"`, run the upload cell and pick a
  zip of your `data/` tree (or upload files into `data/<participant>/` yourself).
- **Colab + Google Drive** — mount Drive and set `DATA_DIR` to your folder there.
- **Local Jupyter** — set `DATA_DIR` to the repo's `data/` folder.

> Files placed directly in `DATA_DIR` (no participant subfolder) are still handled —
> they are grouped under a single participant named after `DATA_DIR`.

In [ ]:
DATA_DIR = "data"          # folder containing the participant subfolders
TIME_UNIT = "s"            # timestamps are in seconds
SMOOTH_WINDOW = 11         # Savitzky-Golay window (odd, in frames) for differentiation; 0 disables

os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
# --- Optional: upload a zip of the data tree in Colab ---------------------
# Uncomment in Colab. Upload a .zip whose top level is your participant folders;
# it is extracted into DATA_DIR.
#
# from google.colab import files
# import zipfile, io
# up = files.upload()
# for name, content in up.items():
#     if name.lower().endswith(".zip"):
#         zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
#     else:                                   # a loose .igs.mha -> put it under 'uploaded/'
#         os.makedirs(os.path.join(DATA_DIR, "uploaded"), exist_ok=True)
#         open(os.path.join(DATA_DIR, "uploaded", name), "wb").write(content)
#
# --- Optional: mount Google Drive instead ---------------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/your_folder"

In [ ]:
def discover_participants(root):
    "Return ordered dict {participant_id: [sorted trial file paths]}."
    parts = {}
    subdirs = sorted(d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d))
    for d in subdirs:
        f = sorted(glob.glob(os.path.join(d, "*.igs.mha")))
        if f:
            parts[os.path.basename(d)] = f
    loose = sorted(glob.glob(os.path.join(root, "*.igs.mha")))   # files directly in DATA_DIR
    if loose:
        parts.setdefault(os.path.basename(os.path.abspath(root)), []).extend(loose)
    return parts

participant_files = discover_participants(DATA_DIR)
assert participant_files, (
    f"No .igs.mha files found under {DATA_DIR!r}. Expected participant subfolders "
    "like data/P01/trial1.igs.mha (see the upload cell above).")

print(f"Found {len(participant_files)} participant(s):")
for pid, fs in participant_files.items():
    print(f"  {pid}: {len(fs)} trial(s)")
    for f in fs:
        print(f"      {os.path.basename(f)}")

## 3 · Parse the sequence files

The `.igs.mha` header is a flat list of `Key = Value` lines; per-frame fields are
named `Seq_Frame<idx>_<Field>`. We read tracking, force, timestamps and the fiducial
points — the referenced video is ignored. Every trial is tagged with its participant.

In [ ]:
def parse_mha(path, participant, trial_no):
    header, frames = {}, {}
    with open(path, "r", errors="ignore") as fh:
        for line in fh:
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip()
            m = re.match(r"Seq_Frame(\d+)_(.+)", k)
            if m:
                frames.setdefault(int(m.group(1)), {})[m.group(2)] = v
            else:
                header[k] = v
            if k == "ElementDataFile":   # end of header; pixel data (if any) follows
                break

    idx = sorted(frames)

    def as_mat(fr, key):
        return np.array([float(x) for x in fr[key].split()], float).reshape(4, 4)

    n_pts = int(header.get("BipolarCollectedPointCount", 0))
    fiducials = (np.array([[float(x) for x in header[f"BipolarCollectedPoint{i}"].split()]
                           for i in range(n_pts)]) if n_pts else None)

    timestamps = np.array([float(frames[i]["Timestamp"]) for i in idx])
    force = np.array([[float(x) for x in frames[i]["Force"].split()] for i in idx])  # fx fy fz tx ty tz

    poses = {}
    for inst in INSTRUMENTS:
        key = inst + "TipToWorldTransform"
        if key in frames[idx[0]]:
            poses[inst] = np.stack([as_mat(frames[i], key) for i in idx])            # (N,4,4)

    return dict(participant=participant, trial=trial_no,
                name=os.path.basename(path).replace(".igs.mha", ""),
                label=f"{participant}·T{trial_no}",
                header=header, timestamps=timestamps, force=force,
                poses=poses, fiducials=fiducials)


trials = []                       # flat list of all trials, in participant/trial order
participants = {}                 # participant_id -> list of its trials
for pid, fs in participant_files.items():
    participants[pid] = []
    for t_no, f in enumerate(fs, start=1):
        tr = parse_mha(f, pid, t_no)
        trials.append(tr)
        participants[pid].append(tr)

for tr in trials:
    dur = tr["timestamps"][-1] - tr["timestamps"][0]
    print(f"{tr['label']:<12s} {tr['name']:<40s} {len(tr['timestamps']):5d} frames "
          f"{dur:6.1f} {TIME_UNIT}  instruments={list(tr['poses'])}")

## 4 · Point-wise registration to a common reference frame

Every trial carries the **same 4 physical fiducials** (`BipolarCollectedPoint0..3`).
We compute the rigid transform (rotation + translation, no scaling) that best maps
each trial's fiducials onto a single **reference set** — by default the first trial
of the first participant — using the closed-form SVD solution (Kabsch / Horn's
absolute orientation). Applying that transform to every `*TipToWorldTransform`
position places **all trials of all participants** in one shared frame, so
trajectories are directly comparable.

The per-trial fiducial **RMSE** flags a mislabeled or mis-collected point.

In [ ]:
def rigid_register(src, dst):
    "Least-squares rigid transform (4x4) mapping src points onto dst; returns (G, rmse)."
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    cs, cd = src.mean(0), dst.mean(0)
    H = (src - cs).T @ (dst - cd)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))          # guard against reflection
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    t = cd - R @ cs
    G = np.eye(4); G[:3, :3] = R; G[:3, 3] = t
    rmse = np.sqrt(np.mean(np.sum(((R @ src.T).T + t - dst) ** 2, axis=1)))
    return G, rmse


ref_pts = trials[0]["fiducials"]                    # common reference for every trial

for tr in trials:
    if tr["fiducials"] is not None and ref_pts is not None:
        tr["G"], tr["rmse"] = rigid_register(tr["fiducials"], ref_pts)
    else:
        tr["G"], tr["rmse"] = np.eye(4), np.nan

    tr["pos"] = {}
    for inst, P in tr["poses"].items():
        xyz = P[:, :3, 3]
        tr["pos"][inst] = (tr["G"][:3, :3] @ xyz.T).T + tr["G"][:3, 3]

    t = tr["timestamps"]
    tr["tnorm"] = (t - t[0]) / (t[-1] - t[0])       # duration normalized to [0, 1]

    print(f"{tr['label']:<12s} registration RMSE = {tr['rmse']:.3f} mm")

## 5 · Force magnitude and motion derivatives

- **Force magnitude** `|F| = √(fx² + fy² + fz²)` (invariant to registration).
- **Velocity, acceleration, jerk** are the 1st/2nd/3rd time-derivatives of tip
  position, magnitudes reported. Position is lightly Savitzky-Golay smoothed before
  differentiating (numerical differentiation amplifies tracking noise); derivatives
  use the **actual timestamps**, so units are mm/s, mm/s², mm/s³.

In [ ]:
def _smooth(x, win):
    if not win or win < 3 or win >= len(x):
        return x
    if win % 2 == 0:
        win += 1
    try:
        from scipy.signal import savgol_filter
        return savgol_filter(x, win, 3, axis=0)
    except Exception:                     # fallback: centered moving average
        k = np.ones(win) / win
        return np.stack([np.convolve(x[:, j], k, mode="same") for j in range(x.shape[1])], 1)


def kinematics(pos, t, win):
    pos = _smooth(pos, win)
    vel = np.gradient(pos, t, axis=0)
    acc = np.gradient(vel, t, axis=0)
    jrk = np.gradient(acc, t, axis=0)
    return {"velocity": np.linalg.norm(vel, axis=1),
            "acceleration": np.linalg.norm(acc, axis=1),
            "jerk": np.linalg.norm(jrk, axis=1)}


for tr in trials:
    tr["fmag"] = np.linalg.norm(tr["force"][:, :3], axis=1)          # |force|
    tr["tmag"] = np.linalg.norm(tr["force"][:, 3:], axis=1)          # |torque| (kept for reference)
    tr["kin"] = {inst: kinematics(tr["pos"][inst], tr["timestamps"], SMOOTH_WINDOW)
                 for inst in tr["pos"]}

KIN_UNITS = {"velocity": "mm/s", "acceleration": "mm/s²", "jerk": "mm/s³"}
# a light->dark ramp to distinguish the trials within one participant
def trial_colors(n):
    return plt.cm.cividis(np.linspace(0.15, 0.85, max(n, 1)))
print("Derived force magnitude and velocity/acceleration/jerk for all trials.")

## 6 · Force magnitude — per participant

In [ ]:
n = len(participants)
ncol = min(3, n); nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.2 * nrow),
                         squeeze=False, sharex=True)
for ax, (pid, ptrials) in zip(axes.flat, participants.items()):
    cols = trial_colors(len(ptrials))
    for tr, c in zip(ptrials, cols):
        ax.plot(tr["tnorm"], tr["fmag"], lw=1.0, color=c, label=f"T{tr['trial']}")
    ax.set_title(f"Participant {pid}")
    ax.set_xlabel("normalized time"); ax.set_ylabel("|force|  (N)")
    ax.margins(x=0); ax.legend(title="trial", fontsize=8)
for ax in axes.flat[n:]:
    ax.set_visible(False)
fig.suptitle("Force magnitude  √(fx²+fy²+fz²)  per participant", fontweight="bold")
fig.tight_layout()
plt.show()

## 7 · Velocity, acceleration and jerk — one figure per participant

Rows are the three motion metrics; columns are the instruments. Each line is one of
the participant's trials, against normalized time.

In [ ]:
metrics = ["velocity", "acceleration", "jerk"]
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(len(metrics), len(INSTRUMENTS),
                             figsize=(4.6 * len(INSTRUMENTS), 2.9 * len(metrics)),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for r, metric in enumerate(metrics):
        for c, inst in enumerate(INSTRUMENTS):
            ax = axes[r][c]
            for tr, col in zip(ptrials, cols):
                if inst in tr["kin"]:
                    ax.plot(tr["tnorm"], tr["kin"][inst][metric], lw=0.9, color=col,
                            label=f"T{tr['trial']}" if (r == 0 and c == 0) else None)
            if r == 0:
                ax.set_title(inst, color=INST_COLOR[inst])
            if c == 0:
                ax.set_ylabel(f"{metric}\n({KIN_UNITS[metric]})")
            if r == len(metrics) - 1:
                ax.set_xlabel("normalized time")
            ax.margins(x=0)
    axes[0][0].legend(title="trial", fontsize=8, loc="upper right")
    fig.suptitle(f"Motion derivatives — participant {pid}", fontweight="bold")
    fig.tight_layout()
    plt.show()

## 8 · 3D instrument trajectories — one figure per participant

Rows are the participant's trials, columns are the instruments. Each registered tip
path is drawn as a 3D line colored from the start (dark) to the end (yellow) of the
recording. All trials share the common registered frame.

In [ ]:
def _color_line3d(ax, xyz, tnorm, lw=1.6):
    pts = xyz.reshape(-1, 1, 3)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = Line3DCollection(segs, cmap=TIME_CMAP, array=tnorm[:-1], linewidth=lw)
    ax.add_collection3d(lc)
    return lc

for pid, ptrials in participants.items():
    nrow, ncol = len(ptrials), len(INSTRUMENTS)
    fig = plt.figure(figsize=(5.6 * ncol, 4.6 * nrow))
    lc = None
    for r, tr in enumerate(ptrials):
        for c, inst in enumerate(INSTRUMENTS):
            ax = fig.add_subplot(nrow, ncol, r * ncol + c + 1, projection="3d")
            if inst not in tr["pos"]:
                ax.set_axis_off(); continue
            P = tr["pos"][inst]
            lc = _color_line3d(ax, P, tr["tnorm"])
            ax.set_title(f"T{tr['trial']} · {inst}", color=INST_COLOR[inst], fontsize=10)
            ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)"); ax.set_zlabel("z (mm)")
            rng = (np.ptp(P, axis=0).max() / 2) or 1
            mid = P.mean(0)
            for setlim, m in zip((ax.set_xlim, ax.set_ylim, ax.set_zlim), mid):
                setlim(m - rng, m + rng)
            ax.view_init(elev=20, azim=-60)
    if lc is not None:
        cb = fig.colorbar(lc, ax=fig.axes, shrink=0.5, pad=0.02)
        cb.set_label("normalized time")
    fig.suptitle(f"Registered tip trajectories — participant {pid}", fontweight="bold")
    plt.show()

## 9 · Summative figures

Averages aggregated across each participant's trials.

- **Per participant**: for one participant, average velocity / acceleration / jerk per
  instrument, plus average force per trial.
- **Cross-participant comparison**: grouped bars (x = participant) for average force
  and for average velocity / acceleration / jerk per instrument. Error bars are the
  standard deviation across that participant's trials (clipped at zero, since the
  quantities are non-negative).

In [ ]:
def _grouped_bar(ax, participant_ids, series, ylabel, title, colors=None):
    "series: dict label -> (means[np], errs[np]); one group of bars per participant."
    x = np.arange(len(participant_ids))
    k = len(series)
    w = 0.8 / k
    for i, (lab, (means, errs)) in enumerate(series.items()):
        means, errs = np.asarray(means, float), np.asarray(errs, float)
        yerr = np.vstack([np.minimum(errs, means), errs])
        off = (i - (k - 1) / 2) * w
        ax.bar(x + off, means, width=w * 0.95, yerr=yerr, label=lab,
               color=None if colors is None else colors[i],
               error_kw=dict(lw=1, capsize=3, ecolor="#555"))
    ax.set_xticks(x); ax.set_xticklabels(participant_ids)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.set_xlabel("participant")
    ax.margins(y=0.15)
    if k > 1:
        ax.legend(fontsize=8)

pids = list(participants)

# per-instrument mean/std across each participant's trials
def inst_stats(metric):
    means, errs = {}, {}
    for inst in INSTRUMENTS:
        m, e = [], []
        for pid in pids:
            vals = [tr["kin"][inst][metric].mean() for tr in participants[pid] if inst in tr["kin"]]
            m.append(np.mean(vals) if vals else 0.0)
            e.append(np.std(vals) if len(vals) > 1 else (np.std(participants[pid][0]["kin"][inst][metric])
                                                         if vals else 0.0))
        means[inst], errs[inst] = m, e
    return means, errs

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) average force magnitude per participant (mean over trials)
fmeans = [np.mean([tr["fmag"].mean() for tr in participants[pid]]) for pid in pids]
ferrs = [np.std([tr["fmag"].mean() for tr in participants[pid]]) if len(participants[pid]) > 1
         else participants[pid][0]["fmag"].std() for pid in pids]
_grouped_bar(axes[0][0], pids, {"|force|": (fmeans, ferrs)},
             "|force|  (N)", "Average force magnitude per participant", colors=["#0072B2"])

# (b–d) velocity / acceleration / jerk, grouped by instrument
for ax, metric in zip([axes[0][1], axes[1][0], axes[1][1]],
                      ["velocity", "acceleration", "jerk"]):
    means, errs = inst_stats(metric)
    series = {inst: (means[inst], errs[inst]) for inst in INSTRUMENTS}
    _grouped_bar(ax, pids, series, KIN_UNITS[metric],
                 f"Average {metric} per instrument",
                 colors=[INST_COLOR[i] for i in INSTRUMENTS])

fig.suptitle("Cross-participant comparison", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

## 10 · Per-trial statistics table

One row per trial with the key summary metrics, plus a per-participant aggregate
(mean across each participant's trials). Both tables are also written to CSV next to
the data so they can be opened in a spreadsheet or fed into statistical tests.

Per instrument, **path length** is the total distance the registered tip travels
(mm); speed / acceleration / jerk columns are the trial means (and peak speed).

In [ ]:
import pandas as pd

def path_length(P):
    return float(np.sum(np.linalg.norm(np.diff(P, axis=0), axis=1)))

rows = []
for tr in trials:
    row = {
        "participant": tr["participant"],
        "trial": tr["trial"],
        "file": tr["name"],
        "n_frames": len(tr["timestamps"]),
        "duration_s": round(float(tr["timestamps"][-1] - tr["timestamps"][0]), 3),
        "reg_rmse_mm": round(float(tr["rmse"]), 4),
        "force_mean_N": round(float(tr["fmag"].mean()), 5),
        "force_peak_N": round(float(tr["fmag"].max()), 5),
        "torque_mean": round(float(tr["tmag"].mean()), 5),
    }
    for inst in INSTRUMENTS:
        if inst not in tr["pos"]:
            continue
        k = tr["kin"][inst]
        row[f"{inst}_path_mm"] = round(path_length(tr["pos"][inst]), 1)
        row[f"{inst}_speed_mean"] = round(float(k["velocity"].mean()), 2)
        row[f"{inst}_speed_peak"] = round(float(k["velocity"].max()), 2)
        row[f"{inst}_accel_mean"] = round(float(k["acceleration"].mean()), 2)
        row[f"{inst}_jerk_mean"] = round(float(k["jerk"].mean()), 1)
    rows.append(row)

per_trial = pd.DataFrame(rows)

# per-participant aggregate: mean of the numeric per-trial metrics
num_cols = [c for c in per_trial.columns if c not in ("participant", "trial", "file", "n_frames")]
per_participant = (per_trial.groupby("participant")[num_cols]
                   .mean().round(3).reset_index())
per_participant.insert(1, "n_trials",
                       per_trial.groupby("participant").size().values)

# save alongside the data
out_trial = os.path.join(DATA_DIR, "metrics_per_trial.csv")
out_part = os.path.join(DATA_DIR, "metrics_per_participant.csv")
per_trial.to_csv(out_trial, index=False)
per_participant.to_csv(out_part, index=False)
print("wrote", out_trial, "and", out_part)

from IPython.display import display
print("\nPer-trial statistics:")
display(per_trial)
print("Per-participant aggregate (mean over trials):")
display(per_participant)

---
### Notes & assumptions

- **Data layout.** `DATA_DIR/<participant>/<trial>.igs.mha`. Trials are numbered by
  filename order within each participant folder. Files sitting directly in `DATA_DIR`
  are grouped under one participant named after the folder.
- **Single reference frame.** All trials of all participants are registered onto
  `trials[0]`'s fiducials, so everything is comparable. Change which trial is the
  anchor by reordering, or set `ref_pts` yourself in §4.
- **Fiducial order matters.** Registration assumes `BipolarCollectedPoint0..3`
  correspond across trials; a high RMSE in §4 usually means a swapped/mis-collected point.
- **Smoothing.** `SMOOTH_WINDOW` sets the Savitzky-Golay window used before
  differentiation. Larger = smoother but more lag; `0` differentiates the raw track.
- **Force is a single sensor**, not per-instrument, so it is summarized per
  participant/trial. Torque magnitude is also parsed (`tr["tmag"]`) if you want it.